## 2.1 Imputing, Encoding and First Pipeline

In [25]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, classification_report
from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from features import *
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report


data = pd.read_csv('train.csv')
child_feature(data)
family_size_feature(data)
name_feature(data)
group_rare_title(data)

In [26]:
X = data.drop(columns=['Survived', 'Ticket', 'PassengerId', 'Cabin', 'Name'])
y = data['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Child,FamilySize,Title
0,3,male,22.0,1,0,7.2500,S,0,2,Mr
1,1,female,38.0,1,0,71.2833,C,0,2,Mrs
2,3,female,26.0,0,0,7.9250,S,0,1,Miss
3,1,female,35.0,1,0,53.1000,S,0,2,Mrs
4,3,male,35.0,0,0,8.0500,S,0,1,Mr


### Categorical and numerical features

In [27]:
cat_features = list(X.drop(columns=X.select_dtypes(include=['int', 'float'])))
num_features = list(X.drop(columns=X.select_dtypes(include=['str'])))

### Numerical and categorical pipeline + Imputer

In [28]:
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

### ColumnTransfer

In [29]:
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

### Final Pipeline

In [30]:
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(C=1.0, max_iter=1000))
])

final_pipeline.fit(X_train, y_train)
prediction = final_pipeline.predict(X_test)

### Confusion Matrix, predict_proba, precision, recall

In [31]:
conf_matrix = confusion_matrix(y_test, prediction)
print(conf_matrix)

proba_survived = final_pipeline.predict_proba(X_test)[:, 1]
new_labels = (proba_survived >= 0.7).astype(int)

new_conf_matrix = confusion_matrix(y_test, new_labels)
print(new_conf_matrix)

[[89 16]
 [21 53]]
[[96  9]
 [34 40]]


### Important to mention:
 - 1. We have to increase recall because it is better to indentify person as dead and he is not than identify person as alive and he is dead

### Cross Validation

In [32]:
cross_validation = cross_val_score(final_pipeline, X, y, cv=5)
print(np.mean(cross_validation))

crv_predict = cross_val_predict(final_pipeline, X, y, cv=5)
conf_matrix = confusion_matrix(y, crv_predict)
print(conf_matrix)

class_report = classification_report(y, crv_predict)
print(class_report)

0.8215491808423827
[[480  69]
 [ 90 252]]
              precision    recall  f1-score   support

           0       0.84      0.87      0.86       549
           1       0.79      0.74      0.76       342

    accuracy                           0.82       891
   macro avg       0.81      0.81      0.81       891
weighted avg       0.82      0.82      0.82       891



### Class imbalance affects performance

 According to classification report we can see that results from the 0 class (death people) is higher than 1 class (survived)acros all three metrics - precision (0.83 vs 0.76), recall (0.86 vs 0.71), f1-score (0.84 vs 0.73). This is caused by more instances from class 0 (549 vs 342).